## 0. Import Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder

import joblib

from pathlib import Path

INTERIM_DIR = Path("../data/interim")

modelling = pd.read_parquet(INTERIM_DIR / "modelling_dataset.parquet")

## 1. Data-quality cleaning

### 1.1 Age

- 18–100 validity range
- invalid age → NaN

In [2]:
modelling["age"] = modelling["age"].where(modelling["age"].between(18, 100), np.nan)

In [3]:
print(modelling["age"].describe())
print(modelling["age"].isna().sum())

count    196803.000000
mean         46.785334
std          15.614404
min          18.000000
25%          34.000000
50%          46.000000
75%          59.000000
max         100.000000
Name: age, dtype: float64
3236


### 1.2 Issue-Redeem-Delay Reengineering

1. If redeem occurs after issue → retain the actual delay.
2. If redeem and issue occur on the same calendar day and within the same hour → set delay to 0.
3. If the delay is substantially negative → treat it as a data-quality issue and set it to missing.

In [4]:
issue = modelling["first_issue_date"]
redeem = modelling["first_redeem_date"]

raw_delay = redeem - issue

same_day = issue.dt.normalize() == redeem.dt.normalize()
within_hour = raw_delay.abs() <= pd.Timedelta(hours=1)

modelling["issue_redeem_delay"] = (
    raw_delay.dt.total_seconds() / (24 * 60 * 60)
)

# Treat small timestamp discrepancies within the same day/hour as simultaneous
modelling.loc[
    (modelling["issue_redeem_delay"] < 0)
    & same_day
    & within_hour,
    "issue_redeem_delay"
] = 0

# Flag remaining negative values as potential data-quality issues
modelling["issue_redeem_delay_issue"] = (
    modelling["issue_redeem_delay"] < 0
)

In [5]:
# inspect data-quality issues 
modelling.loc[
    modelling["issue_redeem_delay_issue"],
    [
        "client_id",
        "first_issue_date",
        "first_redeem_date",
        "issue_redeem_delay"
    ]
]

,client_id,first_issue_date,first_redeem_date,issue_redeem_delay
946,01440dcc9f,2018-11-09 13:50:09,2018-09-27 17:35:20,-42.843623


In [6]:
# Set delay to missing for rows flagged as data-quality issues
modelling.loc[
    modelling["issue_redeem_delay_issue"] & modelling["issue_redeem_delay"].notna(),
    "issue_redeem_delay"
] = np.nan

In [7]:
print(modelling["issue_redeem_delay"].describe())
print(modelling["issue_redeem_delay"].isna().sum())

count    182492.000000
mean        180.531992
std         156.676848
min           0.000000
25%          62.213709
50%         132.254392
75%         252.948142
max         935.570451
Name: issue_redeem_delay, dtype: float64
17547


- 17546 missing because there is no redeem date
- 1 extra because of data error

## 2. Train/ Validation/ Test Split

**Stratified random split on treatment assignment.**

Target:
| Split      | Size | Treated | Control |
| ---------- | ---: | ------: | ------: |
| Train      |  50% |    ~50% |    ~50% |
| Validation |  20% |    ~50% |    ~50% |
| Test       |  30% |    ~50% |    ~50% |


In [8]:
# First: 50% train, 50% temporary
train, temp = train_test_split(
    modelling,
    test_size=0.50,
    stratify=modelling["treatment_flg"],
    random_state=42
)

# Second: split remaining 50% into 20% validation and 30% test
# => validation = 40% of temp, test = 60% of temp
validation, test = train_test_split(
    temp,
    test_size=0.60,
    stratify=temp["treatment_flg"],
    random_state=42
)

In [9]:
# Balance check
for name, data in {
    "full": modelling,
    "train": train,
    "validation": validation,
    "test": test
}.items():
    print(
        name,
        len(data),
        data["treatment_flg"].mean().round(4)
    )

full 200039 0.4998
train 100019 0.4998
validation 40008 0.4998
test 60012 0.4998


### 2.1 Split check: Covariates distributions between treatment and control

In [10]:
# Compare some of the covariate distributions between treatment and control within each split
covariates = [
    "age",
    "gender",
    "n_transactions",
    "n_distinct_products",
    "n_stores",
    "loyalty_points_earned",
    "loyalty_points_spent",
    "spending_last_30d",
    "spending_last_60d",
    "spending_last_90d",
    "total_spending",
    "issue_redeem_delay",
    "avg_basket_value",
    "avg_items_per_basket",
]

#### Numerical covariates

In [11]:
# For numerical covariates: standardized mean difference SMD 

def standardized_mean_difference(data, variable, treatment="treatment_flg"):
    treated = data.loc[data[treatment] == 1, variable].dropna()
    control = data.loc[data[treatment] == 0, variable].dropna()

    pooled_sd = np.sqrt(
        (treated.var(ddof=1) + control.var(ddof=1)) / 2
    )

    if pooled_sd == 0:
        return 0.0

    return (treated.mean() - control.mean()) / pooled_sd

In [13]:
splits = {
    "train": train,
    "validation": validation,
    "test": test
}

balance_results = []

for split_name, data in splits.items():
    for variable in covariates:
        if pd.api.types.is_numeric_dtype(data[variable]):
            smd = standardized_mean_difference(data, variable)

            balance_results.append({
                "split": split_name,
                "variable": variable,
                "smd": smd,
                "abs_smd": abs(smd)
            })

balance_df = pd.DataFrame(balance_results)

balance_df.round(4)

,split,variable,smd,abs_smd
0,train,age,0.0131,0.0131
1,train,n_transactions,0.0044,0.0044
2,train,n_distinct_products,-0.0006,0.0006
3,train,n_stores,-0.0004,0.0004
4,train,loyalty_points_earned,-0.0020,0.0020
5,train,loyalty_points_spent,-0.0048,0.0048
6,train,spending_last_30d,-0.0058,0.0058
7,train,spending_last_60d,-0.0055,0.0055
8,train,spending_last_90d,-0.0021,0.0021
9,train,total_spending,-0.0005,0.0005


| abs(SMD) | Interpretation |
|-----|---------------:|
| < 0.10 | Good/negligible imbalance |
| 0.10–0.20 | Some imbalance; inspect |
| > 0.20 | Potentially meaningful imbalance |

Here all well balanced!

#### Categorical covariates

In [14]:
# For categorical covariates: category proportions 
def categorical_balance(data, variable, treatment="treatment_flg"):
    proportions = (
        data.groupby(treatment)[variable]
        .value_counts(normalize=True)
        .rename("proportion")
        .reset_index()
    )

    return proportions

In [16]:
categorical_balance(train, "gender").round(4)

,treatment_flg,gender,proportion
0,0,U,0.4659
1,0,F,0.3659
2,0,M,0.1682
3,1,U,0.4648
4,1,F,0.3682
5,1,M,0.1670


In [17]:
categorical_balance(validation, "gender").round(4)

,treatment_flg,gender,proportion
0,0,U,0.4574
1,0,F,0.3738
2,0,M,0.1687
3,1,U,0.4637
4,1,F,0.3699
5,1,M,0.1664


In [18]:
categorical_balance(test, "gender").round(4)

,treatment_flg,gender,proportion
0,0,U,0.4671
1,0,F,0.3664
2,0,M,0.1664
3,1,U,0.4612
4,1,F,0.3704
5,1,M,0.1684


All well balanced!

## 3. Preprocessing
*Fitted on the training set only!*   

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder

import joblib

### 3.1 Variable groups 
- numerical log-transformed & imputed
- numerical log-transformed
- numerical non log-transformed & imputed
- categorical

In [20]:
log_impute_vars = [
    "loyalty_points_spent",
    "loyalty_points_earned",
    
    "spending_last_30d",
    "spending_last_60d",
    "spending_last_90d",
    "total_spending",
    
    "n_transactions",
    "transactions_last_30d",
    "transactions_last_60d",
    "transactions_last_90d",
    "n_distinct_products",
    "n_stores",
    
    "median_basket_value",
    "avg_basket_value",
    "median_items_per_basket",
    "avg_items_per_basket",
]

log_structural_vars =[    
    "min_days_between_purchases",
    "median_days_between_purchases",
    "mean_days_between_purchases",
    "std_days_between_purchases",
    "max_days_between_purchases",
    ]

In [21]:
numeric_non_log = [
    "age",
    "observation_period_days",
    "observation_period_months",
    "prop_transactions_points_spent",
    "prop_transactions_points_earned",
    "issue_redeem_delay",
]

In [22]:
categorical_vars = ["gender"]

### 3.2 Preprocessing Pipelines

In [23]:
log_impute_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))
])

log_structural_pipeline = Pipeline([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

gender_pipeline = Pipeline([
    ("encoder", OrdinalEncoder())
])

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "log_impute",
            log_impute_pipeline,
            log_impute_vars
        ),
        (
            "log_structural",
            log_structural_pipeline,
            log_structural_vars
        ),
        (
            "numeric",
            numeric_pipeline,
            numeric_non_log
        ),
        (
            "gender",
            gender_pipeline,
            categorical_vars
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

### 3.3 Fitting to Data

In [32]:
# Fit preprocessing on training data only
preprocessor.fit(train)

# Transform the three splits
X_train = preprocessor.transform(train)
X_validation = preprocessor.transform(validation)
X_test = preprocessor.transform(test)

### 3.4 Exporting the Data

In [33]:
# Get the transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Convert them to DataFrames
train_processed = pd.DataFrame(
    X_train,
    columns=feature_names,
    index=train.index
)

validation_processed = pd.DataFrame(
    X_validation,
    columns=feature_names,
    index=validation.index
)

test_processed = pd.DataFrame(
    X_test,
    columns=feature_names,
    index=test.index
)

# Add treatment and outcome
for X, original in [
    (train_processed, train),
    (validation_processed, validation),
    (test_processed, test)
]:
    X["treatment_flg"] = original["treatment_flg"].values
    X["target"] = original["target"].values

In [34]:
processed_dir = Path("../data/processed")
models_dir = Path("../results/models")

processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

train_processed.to_parquet(
    processed_dir / "train.parquet",
    index=False
)

validation_processed.to_parquet(
    processed_dir / "validation.parquet",
    index=False
)

test_processed.to_parquet(
    processed_dir / "test.parquet",
    index=False
)

# Saving the fitted Preprocesser 
joblib.dump(
    preprocessor,
    models_dir / "preprocessor.joblib"
)

['../results/models/preprocessor.joblib']

### 3.5 Re-Checking RCT Balance 
To confirm that preprocessing didn't introduce an unintended difference between treatment and control

In [37]:
splits_processed = {
    "train": train_processed,
    "validation": validation_processed,
    "test": test_processed
}

balance_results_processed = []

for split_name, data in splits_processed.items():
    for variable in covariates:
        if pd.api.types.is_numeric_dtype(data[variable]):
            smd = standardized_mean_difference(data, variable)

            balance_results_processed.append({
                "split": split_name,
                "variable": variable,
                "smd": smd,
                "abs_smd": abs(smd)
            })

balance_df_processed = pd.DataFrame(balance_results_processed)

balance_df_processed.round(4)

,split,variable,smd,abs_smd
0,train,age,0.0130,0.0130
1,train,gender,-0.0038,0.0038
2,train,n_transactions,0.0064,0.0064
3,train,n_distinct_products,0.0031,0.0031
4,train,n_stores,0.0003,0.0003
5,train,loyalty_points_earned,0.0017,0.0017
6,train,loyalty_points_spent,0.0052,0.0052
7,train,spending_last_30d,0.0021,0.0021
8,train,spending_last_60d,-0.0024,0.0024
9,train,spending_last_90d,0.0018,0.0018


All well balanced!